### Brownian motion and related processes

In [ ]:
import numpy as np

import random
from scipy.stats import norm

import plotly.graph_objects as go

Try naive construction of Brownian motion: $X_t = \sqrt{t}Z_t$

In [ ]:
t = np.linspace(0,1,1001)
zs = norm.rvs(size = 1001)
xt = np.sqrt(t)*zs

plot = go.Figure()
graph = go.Scatter(x = t, y = xt)
plot.add_trace(graph)
plot.show()

For the next problems use the following function for Brownian motion $(W_t)_{t\ge 0}$ construction:

In [ ]:
def bm_simulations(n_paths, granularity, max_time):
    n_steps = granularity * max_time
    
    paths = []
    for i in range(n_paths):
        xs = random.choices([-1, 1], weights = [0.5, 0.5], k = n_steps)
        xs = [0] + xs
        path = np.cumsum(xs)/np.sqrt(granularity)
        paths.append(path)
        
    return np.array(paths)

Brownian bridge: model process $B_t = W_t-tW_1,\,t\in [0,1]$.

In [ ]:
def bb_simulations(n_paths, n):
    bm_paths = bm_simulations(n_paths, n, 1)
    t = np.linspace(0, 1, n+1)
    
    bb_paths = []
    for bm_path in bm_paths:
        bb_path = bm_path - t*bm_path[-1]
        bb_paths.append(bb_path)
        
    return [t, np.array(bb_paths)]

In [ ]:
N = 50
n = 1000
trajectories = bb_simulations(N, n)
time = trajectories[0]
paths = trajectories[1]

plot = go.Figure()
for path in paths:
    graph = go.Scatter(x = time, y = path)
    plot.add_trace(graph)
plot.show()

Check expectation $EB_{0.5}$ and covariance $cov(B_{0.3}, B_{0.5})$.

In [ ]:
n = 100
N = 10000
trajectories = bb_simulations(N, n)
time = trajectories[0]
paths = trajectories[1]
B_05 = paths[:,n//2]
mean_ = np.mean(B_05)

B_03 = paths[:,30]
covariance = np.cov(B_03, B_05)
cov = covariance[0][1]
print(cov, abs(cov - (min(0.3, 0.5) - 0.3*0.5)))

### Geometric Brownian motion

Construct geometric Bm using representation
$$
S_t = S_0e^{aU+bt},\ U\sim Binom(t, q)
$$

In [ ]:
T = 2
sigma = 0.5
r = 0.2
S0 = 100

m = 1000
u = 1 + sigma/np.sqrt(m)
d = 1 - sigma/np.sqrt(m)
r_m = r/m 
q = (1+r_m-d)/(u-d)
a = np.log(u/d)
b = np.log(d)

steps = random.choices([0, 1], weights = [1-q, q], k = m*T+1)
t = np.linspace(0, T, m*T+1)
St = [S0]
for i in range(1, m*T+1):
    Si = S0*np.exp(a*sum(steps[0:i])+b*i)
    St.append(Si)

plot = go.Figure()
graph = go.Scatter(x = t, y = St)
plot.add_trace(graph)
plot.show()

In [ ]:
def geom_bm_simulations(S0, sigma, r, n_paths, granularity, max_time):
    u = 1 + sigma/np.sqrt(granularity)
    d = 1 - sigma/np.sqrt(granularity)
    r_m = r/granularity 
    q = (1+r_m-d)/(u-d)
    a = np.log(u/d)
    b = np.log(d)
   
    paths = []
    n_steps = max_time*granularity+1
    for _ in range(n_paths):
        steps = random.choices([0, 1], weights = [1-q, q], k = n_steps)
        path = [S0]
        for i in range(1, n_steps):
            Si = S0*np.exp(a*sum(steps[0:i])+b*i)
            path.append(Si)
        paths.append(np.array(path))
    
    return np.array(paths)

In [ ]:
T = 2
sigma = 0.1
r = 0.2
S0 = 100
m = 1000
N = 35
print(r - sigma**2/2, u, d)

t = np.linspace(0, T, m*T+1)
paths = geom_bm_simulations(S0, sigma, r, N, m, T)

plot = go.Figure()
for path in paths:
    graph = go.Scatter(x = t, y = path)
    plot.add_trace(graph)
plot.show()

### Stochastic differential equations

Simulate Geometric Brownian motion using SDE (Stochastic differential equations) approach<br>
$$
dS_t=rS_tdt + \sigma S_t dW_t
$$
On a whiteboard derive expectation $\mathsf{E}(S_t).$

In [ ]:
T = 2
r = 0.2
sigma = 0.5
m = 1000
h = 1/m
S0 = 3

ksi = norm.rvs(0, np.sqrt(h), size = m*T)
path = [S0]
for i in range(m*T):
    Si = path[-1] + r*h*path[-1] + sigma*path[-1]*ksi[i]
    path.append(Si)

t = np.linspace(0, T, m*T+1)
plot = go.Figure()
graph = go.Scatter(x = t, y = path)
plot.add_trace(graph)
plot.show()